In [18]:
import json
import os

path = "similar_account_data.json"

if os.path.getsize(path) == 0:
    print("File is empty")
    data = {}
else:
    with open(path, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            print("Invalid JSON, showing first 500 chars:")
            f.seek(0)
            print(f.read(500))
            raise

In [31]:
data.keys()

dict_keys(['46869721446', '9267995', '6692819958', '27007247969', '4624808342', '46386038482', '7622518786', '36750245', '39810477904', '6773171', '49976737', '56205933618', '52447928269', '6629526550', '47472417177', '51191470442', '29779728', '313973016', '13032805342', '900304285', '18422054', '47322257697', '11417138358', '7867189192', '2171052335', '63942150624', '7744420442', '6247412499', '1731240429', '1766778903', '39383914159', '275945033', '11398391808', '11607669618', '4087924939', '36612287375', '576431365', '56553654392', '63880726161', '34112699221', '64198315013', '21937916588', '51969981389', '48452429500', '6484963024', '46445660106', '10607551224'])

In [41]:
import requests
from pathlib import Path
from tqdm import tqdm

for account_id in data.keys():
    for post_idx in range(len(data[account_id]['latest_posts'])):
    
        url = data[account_id]['latest_posts'][post_idx]["image_url"]
        
        out = Path(f"images/{account_id}_post_{post_idx}.jpg")
        out.parent.mkdir(exist_ok=True)
        
        with requests.get(url, stream=True, timeout=30) as r:
            r.raise_for_status()
            with open(out, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        
        print("Saved:", out)

Saved: images/46869721446_post_0.jpg
Saved: images/46869721446_post_1.jpg
Saved: images/46869721446_post_2.jpg
Saved: images/46869721446_post_3.jpg
Saved: images/46869721446_post_4.jpg
Saved: images/46869721446_post_5.jpg
Saved: images/46869721446_post_6.jpg
Saved: images/46869721446_post_7.jpg
Saved: images/46869721446_post_8.jpg
Saved: images/46869721446_post_9.jpg
Saved: images/46869721446_post_10.jpg
Saved: images/46869721446_post_11.jpg
Saved: images/9267995_post_0.jpg
Saved: images/9267995_post_1.jpg
Saved: images/9267995_post_2.jpg
Saved: images/9267995_post_3.jpg
Saved: images/9267995_post_4.jpg
Saved: images/9267995_post_5.jpg
Saved: images/9267995_post_6.jpg
Saved: images/9267995_post_7.jpg
Saved: images/9267995_post_8.jpg
Saved: images/9267995_post_9.jpg
Saved: images/9267995_post_10.jpg
Saved: images/9267995_post_11.jpg
Saved: images/6692819958_post_0.jpg
Saved: images/6692819958_post_1.jpg
Saved: images/6692819958_post_2.jpg
Saved: images/6692819958_post_3.jpg



KeyboardInterrupt



In [38]:
for post_idx in range(len(data[account_id]['latest_posts'])):
    print(post_idx, data[account_id]['latest_posts'][post_idx]['like_count'])

0 1889
1 2129
2 5381
3 2951
4 2147
5 4420
6 5255
7 14221
8 4395
9 23392
10 10693
11 9804


In [44]:
# acync download
import asyncio
import aiohttp
from pathlib import Path
from tqdm.asyncio import tqdm

CONCURRENCY = 50
OUTPUT_DIR = Path("images")
OUTPUT_DIR.mkdir(exist_ok=True)

async def download_image(session, url, path, sem):
    async with sem:
        try:
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=30)) as resp:
                resp.raise_for_status()
                content = await resp.read()

            with open(path, "wb") as f:
                f.write(content)

        except Exception as e:
            print("Failed:", url, e)

async def main():
    tasks = []
    sem = asyncio.Semaphore(CONCURRENCY)

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.instagram.com/"
    }

    async with aiohttp.ClientSession(headers=headers) as session:

        for account_id in data.keys():
            posts = data[account_id]["latest_posts"]

            for post_idx, post in enumerate(posts):

                url = post["image_url"]
                if url is None:
                    continue

                out = OUTPUT_DIR / f"{account_id}_post_{post_idx}.jpg"

                tasks.append(download_image(session, url, out, sem))

        for f in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
            await f

In [43]:
await main()

100%|██████████| 563/563 [00:05<00:00, 105.63it/s]
